# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` per the Silver to Gold STTM: one row per hospital per calendar day. It aggregates `silver.lab_results` by hospital_id and test_date, LEFT JOINs hospitals and calendar, and adds a day-over-day LAG trend. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

Note: the STTM names the calendar key `date_key`, but the column in `silver.calendar` is actually `datekey` (an int in yyyyMMdd format), so the join uses `datekey`.

In [0]:
dbutils.widgets.text("catalog", "caresync_databricks")
CATALOG = dbutils.widgets.get("catalog")

SILVER = f"{CATALOG}.silver"
TARGET_TABLE = f"{CATALOG}.gold.hospital_daily_summary"
print(f"Source: {SILVER}  ->  Target: {TARGET_TABLE}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE}
COMMENT 'Gold: one row per hospital per day with lab-test volumes, abnormal/critical rates and day-over-day trend. Source: STTM Silver to Gold v1.0.'
AS
WITH daily AS (
    SELECT
        hospital_id,
        test_date                             AS summary_date,
        COUNT(*)                              AS total_tests,
        COUNT(DISTINCT patient_id)            AS unique_patients_tested,
        COUNT(DISTINCT doctor_id)             AS active_doctor_count,
        COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')  AS critical_test_count
    FROM {SILVER}.lab_results
    GROUP BY hospital_id, test_date
),
with_lag AS (
    SELECT
        d.*,
        LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
    FROM daily d
)
SELECT
    CAST(w.hospital_id AS INT)                                                     AS hospital_id,
    h.hospital_name,
    w.summary_date,
    c.day_name,
    c.is_weekend,
    CAST(w.total_tests AS INT)                                                     AS total_tests,
    CAST(w.unique_patients_tested AS INT)                                          AS unique_patients_tested,
    CAST(w.active_doctor_count AS INT)                                             AS active_doctor_count,
    CAST(w.abnormal_test_count AS INT)                                             AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                             AS critical_test_count,
    CAST(ROUND(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(ROUND(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2))  AS critical_rate_pct,
    CAST(w.prev_day_total_tests AS INT)                                            AS prev_day_total_tests,
    CASE
        WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
        ELSE CAST(ROUND((w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests, 2) AS DECIMAL(6,2))
    END                                                                            AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {SILVER}.hospitals h
       ON w.hospital_id = h.hospital_id
LEFT JOIN {SILVER}.calendar c
       ON c.datekey = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

print(f"Written {spark.table(TARGET_TABLE).count():,} rows to {TARGET_TABLE}")

In [0]:
import json

record_count = spark.table(TARGET_TABLE).count()

# Returned as the notebook output (visible to Jobs / ADF / dbutils.notebook.run callers)
# dbutils.notebook.exit(json.dumps({"table": TARGET_TABLE, "record_count": record_count}))
dbutils.notebook.exit(str(record_count))